# Performance Pipeline — AWS Connect schema migration

**Context.** The `input_performance` raw source ("New Look Excel" folder) now ships an AWS Connect–style
export (file pattern e.g. `AWS 2026_06`) with a much narrower, renamed column set (see the column list
the user supplied). The standalone **Survey Dump** and **Delayed Closure** raw sources are broken and are
fully removed from this pipeline — none of their derived metrics (DUET/NPS/Exceed Chat/Ghost/etc.) are
computed anymore. **T3** has also been removed per request (no more `T3_INPUT` / `t3_final`, no more `T3`
column or join).

**Design choice.** Rather than rewriting every downstream reference, a small adapter block right after
`PERFORMANCE_INPUT` is loaded renames/derives the few AWS columns that are actually consumed downstream
back to the legacy canonical names (`Joined Time`, `Handle Time (Sum)`, `Talk Time (Sum)`,
`Wrap Up Time (Sum)`, `Agent Business Location`, `Initiated Outbound (Yes / No)`, `Latest VA Product`,
`Latest VA Intent`, `Language`). Everything past that point (HC join, IEX/night-shift, AON, LC threshold,
PST/VNT interval calc, AFCR/Re-Direct joins) is untouched.

**Please verify on first run:** the diagnostic print right after the adapter block shows
`Connected To Agent Date` / `Connected To Agent Time` next to the parsed `Joined Time`. If the parser
guessed the wrong format, tell me the real one and I'll fix the one function responsible for it.

In [1]:
import polars as pl
import os
import glob
import shutil
import json
from datetime import datetime, timedelta
from typing import Optional, Dict

In [2]:
today = datetime.now().strftime('%b_%d_%Y')

def input_data(folder_path: str, sheet_name: str = None, filename_keyword: str = None) -> pl.DataFrame:
    file_paths = (
        glob.glob(os.path.join(folder_path, "*.xlsx")) +
        glob.glob(os.path.join(folder_path, "*.csv"))
    )
    if filename_keyword:
        kw = filename_keyword.lower()
        file_paths = [f for f in file_paths if kw in os.path.basename(f).lower()]
    df_list = []
    for file in file_paths:
        export_time = datetime.fromtimestamp(os.path.getmtime(file))
        file_name   = os.path.basename(file)

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name, engine='calamine')
            df = df.select(pl.all().cast(pl.String))
        elif file.endswith('.csv'):
            df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0, ignore_errors=True)

        df.columns = [c.strip().replace('\ufeff', '').replace('\u200b', '') for c in df.columns]

        df = df.with_columns(
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time'),
        )
        df_list.append(df)

    if not df_list:
        return pl.DataFrame()
    return pl.concat(df_list, how='diagonal_relaxed')

def input_data_parquet(folder_path: str) -> pl.DataFrame:
    file_paths = glob.glob(os.path.join(folder_path, "*.parquet"))
    if not file_paths:
        return pl.DataFrame()
    lf_list = [
        pl.scan_parquet(file).with_columns(
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(datetime.fromtimestamp(os.path.getmtime(file))).alias('Export Time'),
        )
        for file in file_paths
    ]
    return pl.concat(lf_list, how='vertical').collect()


def csv_to_parquet(input_folder: str, output_folder: str, schema_overrides: Optional[Dict] = None) -> None:
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    if not csv_files:
        return

    if schema_overrides is None:
        # Promoter/Detractor Score overrides removed: those columns no longer
        # exist on any retained CSV source (Survey is gone; AWS performance
        # export doesn't carry them either).
        schema_overrides = {}
    schema_overrides["Itinerary"] = pl.String  # still needed for AFCR/FCR CSVs

    for csv_file in csv_files:
        file_name   = os.path.basename(csv_file)
        output_path = os.path.join(output_folder, os.path.splitext(file_name)[0] + ".parquet")
        try:
            pl.scan_csv(
                csv_file,
                infer_schema_length=10000,
                schema_overrides=schema_overrides,
                ignore_errors=True,
            ).sink_parquet(output_path)
        except Exception as e:
            print(f"Critical failure while converting {file_name} to Parquet: {e}")


def copy_csv_files(source_folder: str, destination_folder: str) -> None:
    os.makedirs(destination_folder, exist_ok=True)
    for f in glob.glob(os.path.join(source_folder, "*.csv")):
        shutil.copy2(f, destination_folder)


def restore_all_folders(backup_folder: str, paths_dict: dict, periods: list, mapping: dict) -> None:
    for key, identifier in mapping.items():
        if key not in paths_dict:
            continue
        dest_folder = paths_dict[key]
        os.makedirs(dest_folder, exist_ok=True)
        for period in periods:
            for file_path in glob.glob(os.path.join(backup_folder, f"*{identifier}*{period}*.csv")):
                shutil.copy2(file_path, dest_folder)


def clean_raw_input_folders(folder_paths: dict, target_periods: list, file_patterns: dict) -> int:
    total_deleted = 0
    for folder_key, pattern in file_patterns.items():
        source_dir = folder_paths.get(folder_key)
        if not source_dir or not os.path.exists(source_dir):
            continue
        for period in target_periods:
            for filepath in glob.glob(os.path.join(source_dir, f"*{pattern}*{period}*.*")):
                try:
                    os.remove(filepath)
                    total_deleted += 1
                except Exception as e:
                    print(f"Could not delete {filepath}: {e}")
    return total_deleted


def pre_process_fcr_excel(folder_path: str) -> None:
    excel_files = (
        glob.glob(os.path.join(folder_path, "*.xlsx")) +
        glob.glob(os.path.join(folder_path, "*.xls"))
    )
    for file_path in excel_files:
        try:
            df = pl.read_excel(file_path, engine="calamine", has_header=False)
            if df.height < 2:
                continue

            # Detect actual header row (scan first 10 rows)
            header_row_index = 0
            for i in range(min(10, df.height)):
                row_data = [str(x).strip() for x in df.row(i) if x is not None]
                if "FCR Category" in row_data or "Conversation Id" in row_data:
                    header_row_index = i
                    break

            # Build deduplicated header list
            seen: set = set()
            headers = []
            for i, c in enumerate(df.row(header_row_index)):
                col_name = str(c).strip() if (c is not None and str(c).strip()) else f"col_{i}"
                if col_name in seen:
                    col_name = f"{col_name}_dup_{i}"
                seen.add(col_name)
                headers.append(col_name)

            df = df.slice(header_row_index + 1)
            df.columns = headers
            df.write_csv(os.path.splitext(file_path)[0] + ".csv")
        except Exception:
            pass

In [3]:
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path  = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

# NOTE: "input_survey" and "input_delayed_closure" removed — both raw sources
# are broken and none of their derived metrics are computed anymore.
# "input_t3" removed — T3 calculation dropped per request.
folder_paths = {
    "input_performance":          f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data',
    "output_performance_combine": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/OUTPUT_PERFORMANCE/OUTPUT_PERFORMANCE_COMBINE',
    "hc_extend_by_month":         f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    "input_survey":               f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_t3":                   f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_delayed_closure":      f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE',
    "input_afcr":                 f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR',
    "input_iex":                  f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX',
    "mapping_file":               f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/AQG_summarized.xlsx',
    "global_hc":                  f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/Global_HC.parquet',
    "input_parquet_performance":  f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/PARQUET',
    "input_csv_pulse":            f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/0_source',
    "output_csv_pulse":           f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/1_des',
    "input_xlsx_qa_audit":        f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Detractor_Compliance/2_qa_audit',
    "input_csv_re_direct":        f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/RE-DIRECT',
    "csv":                        f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/CSV',
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")
print("-" * 60)

pre_process_fcr_excel(folder_paths["input_afcr"])
csv_keys = ["input_performance", "input_t3", "input_csv_re_direct", "input_afcr"]
for k in csv_keys:
    src_folder = folder_paths[k]
    csv_to_parquet(src_folder, folder_paths["input_parquet_performance"])
    copy_csv_files(src_folder, folder_paths["csv"])

--- FULL FOLDER PATHS LIST ---
input_performance: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_EN/new_look_excel_data
output_performance_combine: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/OUTPUT_PERFORMANCE/OUTPUT_PERFORMANCE_COMBINE
hc_extend_by_month: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_survey: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN
input_t3: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN
input_delayed_closure: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/DELAYED_CLOSURE
input_afcr: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/FCR
input_iex: C:/Users/huuchinh.nguyen/Concentrix Corporat

In [4]:
columns_to_sec = ['Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Target']

# Load IEX: deduplicate and cast all columns in a single pass
IEX = (
    input_data(folder_paths["input_iex"])
    .unique()
    .with_columns(
        pl.col(['Date']).str.to_date("%Y-%m-%d", strict=False),
        pl.col(['Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift',
                'Datetime_First_Start_Shift', 'Datetime_First_End_Shift'])
          .str.to_datetime("%Y-%m-%d %H:%M:%S%.f", strict=False),
        # 'Target' removed here — it's handled below in columns_to_sec (*3600)
        pl.col(['Night_Shift', 'Unplanned', 'Planned',
                'Roster Presented', 'Roster Scheduled']).cast(pl.Float64),
        # Convert hour-based columns to seconds (includes Target → Float64 * 3600)
        *[(pl.col(col).fill_null(0).cast(pl.Float64) * 3600).alias(col)
          for col in columns_to_sec],
    )
)

# Build night-shift lookup: each date's own flag + the previous night's flag
Night_Shift_1 = IEX[['Date', 'Email Id', 'Night_Shift']].unique()
Night_Shift   = (
    Night_Shift_1
    .with_columns((pl.col('Date') - pl.duration(days=1)).alias('Previous Date'))
    .join(Night_Shift_1, left_on=['Previous Date', 'Email Id'],
          right_on=['Date', 'Email Id'], how='left')
    .rename({'Night_Shift_right': 'Previous_Night_Shift'})
)
Previous_Date = IEX[['Date', 'Email Id', 'Datetime_First_Start_Shift', 'Shift Tracking']].unique()

In [5]:
# Update to the period currently being processed (matches the new "AWS YYYY_MM" file naming)
target_periods = ["2026-06"]

file_patterns = {
    "input_performance": "AWS",
    "input_t3": "T3",
    "input_csv_re_direct": "re-direct",
    "input_afcr": "FCR",
}

# restore_all_folders(folder_paths["csv"], folder_paths, target_periods, file_patterns)
# clean_raw_input_folders(folder_paths, target_periods, file_patterns)

## Removed: Survey Dump, Delayed Closure & T3

Three raw sources/blocks have been removed:

- **Survey** (`input_survey`) used to feed `survey_ir`, `survey_ae`, `survey_duet`, `verbatim` → NPS (`_promoter`/`_detractor`/`_neutral`/`_nps_type`/`_survey`/`_ir`/`_ae`), DUET (`delight`/`usability`/`ease`/`trust`/`DUET`) and `_verbatim`. None of these are computed anymore.
- **Delayed Closure** (`input_delayed_closure`) used to feed `delayed_closure` → `Exceed Time`/`Exceed Chat`/`Exceed Bucket`/`Agent Disconnect`/`Ghost`/`Requeued`/`Traveler Unresponsive`. None of these are computed anymore.
- **T3** (`input_t3`) removed per request — no more `T3_INPUT`/`t3_final`, no `T3` column, no join on `Conversation Id` against it.
- The AWS performance export also no longer carries `Has Followup Time`, so `CCR72` / `_fup_72` / `_rr` are gone too (no substitute column exists).
- It also no longer carries `Promoter Score (Calc)`, `Detractor Score (Calc)`, `Survey Submitted/Offered (Count)` — those were embedded directly in the old "New Look Excel" export and are simply absent now.

AFCR/FCR and Re-Direct are untouched below — neither was reported as broken.

In [6]:
def process_afcr_folder(folder_path: str) -> pl.DataFrame:
    # Cast numeric columns at read time; no separate with_columns pass needed
    _afcr_schema: Dict[str, pl.PolarsDataType] = {
        "Itinerary":       pl.String,
        "Handle Time":     pl.Float64,
        "Duet":            pl.Float64,
        "Passed Sessions": pl.Int64,
        "Failed Sessions": pl.Int64,
    }
    all_dataframes = []
    for file_path in glob.glob(os.path.join(folder_path, "*.csv")):
        file_name   = os.path.basename(file_path)
        export_time = datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
        df = pl.read_csv(
            file_path,
            encoding="utf-8",
            schema_overrides=_afcr_schema,
            infer_schema_length=10000,
            ignore_errors=True,
        ).with_columns(
            pl.lit(file_name).alias('File Name'),
            pl.lit(export_time).alias('Export Time'),
        )
        all_dataframes.append(df)

    if not all_dataframes:
        return pl.DataFrame()
    return pl.concat(all_dataframes, how="diagonal_relaxed")


afcr_input = process_afcr_folder(folder_paths["input_afcr"])

if not afcr_input.is_empty():
    afcr_input = (
        afcr_input
        .filter(pl.col("Vendor Partner Location") == "Concentrix (Ho Chi Minh City)")
        .select([
            pl.col("Agent Email Address").alias("Agent Email ID"),
            pl.col("Conversation Id"),
            pl.col("Passed Sessions"),
            pl.col("Failed Sessions"),
            pl.when(pl.col("Passed Sessions").is_in([0, 1])).then(1).otherwise(0).alias("Total Sessions"),
        ])
        .unique()
    )

In [7]:
SURVEY_INPUT = input_data(folder_paths["input_survey"], filename_keyword="Survey Dump")

SURVEY_INPUT = SURVEY_INPUT.rename({
    "Conversation_id": "Conversation Id",
    "Agent Email":     "Agent Email ID",
})

# Thêm key_survey ngay sau khi rename
SURVEY_INPUT = SURVEY_INPUT.with_columns(
    pl.concat_str(
        [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
        separator="_"
    ).alias("key_survey")
)

survey_nps = (
    SURVEY_INPUT
    .filter(pl.col("NPS Type").is_not_null())
    .with_columns([
        pl.col("NPS Type").alias("_nps_type"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "promoter").then(1).otherwise(0).alias("_promoter"),
        pl.when(pl.col("NPS Type").str.to_lowercase() == "detractor").then(1).otherwise(0).alias("_detractor"),
        pl.when(pl.col("NPS Type").str.to_lowercase().is_in(["neutral", "passive"])).then(1).otherwise(0).alias("_neutral"),
        pl.lit(1).alias("_survey"),
    ])
    .select(["key_survey", "Conversation Id", "_nps_type", "_promoter", "_detractor", "_neutral", "_survey"])
    .unique()
)

survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("DUET Score Type").is_not_null())
    .with_columns(
        pl.when(pl.col("DUET Score Type").str.to_lowercase().str.contains("positive"))
          .then(1).otherwise(0).alias("DUET")
    )
    .select(["key_survey", "Conversation Id", "DUET"])
    .unique()
)

verbatim = (
    SURVEY_INPUT
    .filter(
        pl.col("Response Text EN").is_not_null() &
        (pl.col("Response Text EN").str.strip_chars() != "")
    )
    .with_columns(
        pl.col("Response Text EN")
          .str.replace_all(r"[\r\n\t]+", " ")
          .str.strip_chars()
          .alias("_verbatim")
    )
    .select(["key_survey", "Conversation Id", "_verbatim"])
    .unique()
)

survey_final = (
    survey_nps
    .join(survey_duet, on=["key_survey", "Conversation Id"], how="left")
    .join(verbatim,    on=["key_survey", "Conversation Id"], how="left")
)

survey_final

key_survey,Conversation Id,_nps_type,_promoter,_detractor,_neutral,_survey,DUET,_verbatim
str,str,str,i32,i32,i32,i32,i32,str
"""vanphuc.bui@concentrix.com_692…","""692df70b-35cb-41ab-85bc-68612c…","""Neutral""",0,0,1,1,0,"""customer service was great; we…"
"""ashish.prasad3@concentrix.com_…","""7c6254f6-d6f3-4642-b68a-21bd07…","""Detractor""",0,1,0,1,0,"""I recently contacted Orbitz tw…"
"""huuan.truong@concentrix.com_3c…","""3c0b5fb1-961e-4dde-bfaf-a3ac3d…","""Detractor""",0,1,0,1,0,"""Get rid of the virtual agents …"
"""thilanhuong.tran1@concentrix.c…","""c40a8a28-dd98-489b-8b87-2a239d…","""Promoter""",1,0,0,1,1,null
"""pooja.verma7@concentrix.com_a2…","""a229a552-3582-4b3f-b74d-11e310…","""Detractor""",0,1,0,1,0,null
…,…,…,…,…,…,…,…,…
"""aqib.nathoo@concentrix.com_277…","""2777ad18-530d-414f-9cf1-22ae30…","""Promoter""",1,0,0,1,1,"""Honestly, I had a wonderful ex…"
"""hoangmaithy.le@concentrix.com_…","""0aa05536-819a-419d-9f4d-bc66cb…","""Promoter""",1,0,0,1,1,null
"""tranthanhphong.nguyen@concentr…","""b950805a-8959-4f42-9150-ffae30…","""Promoter""",1,0,0,1,1,null


In [8]:
T3_INPUT = input_data(folder_paths["input_t3"], filename_keyword="T3_CNX_AWS")

t3_final = (
    T3_INPUT
    .with_columns(
        pl.when(
            pl.col("Transfer Destination").str.contains("Tier 3", literal=True)
        ).then(1).otherwise(0).alias("T3")
    )
    .filter(pl.col("T3") == 1)
    .with_columns(
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_t3")
    )
    .select(["key_t3", "T3"])
    .unique()
)
t3_final

key_t3,T3
str,i32
"""enoch.pardeshi@concentrix.com_…",1
"""rajib.chhatui@concentrix.com_d…",1
"""prerana.sinha1@concentrix.com_…",1
"""omar.ahmedhassan2@concentrix.c…",1
"""shatadru.chowdhury@concentrix.…",1
…,…
"""kritika.thakur@concentrix.com_…",1
"""toshani.sengupta@concentrix.co…",1
"""prem.aklujkar@concentrix.com_2…",1


In [9]:
DELAYED_CLOSURE_INPUT = (
    input_data(folder_paths["input_delayed_closure"], filename_keyword="excess_aws")
    .unique(subset=["User Email", "Conversation ID"], keep="last")
)

delayed_closure = (
    DELAYED_CLOSURE_INPUT
    .select([
        "User Email", "Conversation ID",
        "Excess Time", "Disconnected Reason (groups)",
        "last_traveler_message_sent_datetime_utc",
    ])
    .rename({
        "User Email":      "Agent Email ID",
        "Conversation ID": "Conversation Id",
        "Excess Time":     "_excess_time_raw",
    })
    .with_columns(
        pl.col("_excess_time_raw").cast(pl.Float64).fill_null(0).alias("Exceed Time"),
    )
    .with_columns(
        (pl.col("Exceed Time") > 0).cast(pl.Int8).alias("Exceed Chat"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("agent").cast(pl.Int8).alias("Agent Disconnect"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("ghost").cast(pl.Int8).alias("Ghost"),
        pl.col("Disconnected Reason (groups)").str.to_lowercase()
          .str.contains("requeue").cast(pl.Int8).alias("Requeued"),
        pl.col("last_traveler_message_sent_datetime_utc")
          .is_null().cast(pl.Int8).alias("Traveler Unresponsive"),
        pl.concat_str(
            [pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_delayed_closure"),
    )
    .drop(["_excess_time_raw", "Disconnected Reason (groups)", "last_traveler_message_sent_datetime_utc"])
    .group_by(["Agent Email ID", "Conversation Id", "key_delayed_closure"])
    .agg(
        pl.sum("Exceed Time"),
        pl.sum("Exceed Chat"),
        pl.sum("Agent Disconnect"),
        pl.sum("Ghost"),
        pl.sum("Requeued"),
        pl.sum("Traveler Unresponsive"),
    )
)

_em = pl.col("Exceed Time") / 60
delayed_closure = delayed_closure.with_columns(
    pl.when(pl.col("Exceed Time") <= 0).then(pl.lit(None, dtype=pl.Utf8))
      .when(_em <= 1).then(pl.lit("01 Mins"))
      .when(_em <= 2).then(pl.lit("02 Mins"))
      .when(_em <= 3).then(pl.lit("03 Mins"))
      .when(_em <= 4).then(pl.lit("04 Mins"))
      .when(_em <= 5).then(pl.lit("05 Mins"))
      .when(_em <= 10).then(pl.lit("05-10 Mins"))
      .when(_em <= 15).then(pl.lit("10-15 Mins"))
      .when(_em <= 30).then(pl.lit("15-30 Mins"))
      .otherwise(pl.lit("30+ Min"))
      .alias("Exceed Bucket")
)

delayed_closure

Agent Email ID,Conversation Id,key_delayed_closure,Exceed Time,Exceed Chat,Agent Disconnect,Ghost,Requeued,Traveler Unresponsive,Exceed Bucket
str,str,str,f64,i64,i64,i64,i64,i64,str
"""shatadru.chowdhury@concentrix.…","""d358d32a-b0af-403a-9fc7-be3781…","""shatadru.chowdhury@concentrix.…",296.0,1,1,0,0,0,"""05 Mins"""
"""myuyen.phung@concentrix.com""","""1be8f748-8e99-498f-b2d0-86152f…","""myuyen.phung@concentrix.com_1b…",44.0,1,1,0,0,0,"""01 Mins"""
"""kishan.haldar@concentrix.com""","""5723c67d-70cd-4922-b3e9-283436…","""kishan.haldar@concentrix.com_5…",253.0,1,1,0,0,0,"""05 Mins"""
"""surya.raut@concentrix.com""","""883b61d1-ea85-4f85-8530-58c7d5…","""surya.raut@concentrix.com_883b…",92.0,1,1,0,0,0,"""02 Mins"""
"""thilanhuong.tran1@concentrix.c…","""d8157526-7a03-40ef-8fd2-bb2a7b…","""thilanhuong.tran1@concentrix.c…",3.0,1,1,0,0,1,"""01 Mins"""
…,…,…,…,…,…,…,…,…,…
"""thikimtuyen.phan@concentrix.co…","""cb52aaf6-0700-4ed8-a675-3f3147…","""thikimtuyen.phan@concentrix.co…",62.0,1,1,0,0,0,"""02 Mins"""
"""dinhtuan.nguyen@concentrix.com""","""7e7101e3-5e28-4e93-a530-f81a1b…","""dinhtuan.nguyen@concentrix.com…",182.0,1,1,0,0,0,"""04 Mins"""
"""thingochan.dinh@concentrix.com""","""b57cbdc9-d2a7-46d8-96b7-2d9d22…","""thingochan.dinh@concentrix.com…",67.0,1,1,0,0,0,"""02 Mins"""


In [10]:
RE_DIRECT_INPUT = input_data(folder_paths["input_csv_re_direct"])

re_direct_final = (
    RE_DIRECT_INPUT
    .with_columns(
        Re_Direct=pl.lit(1),
        **{
            "Re-Direct Text": (
                pl.col("Text").cast(pl.Utf8).fill_null("")
                  .str.replace_all(r"(\r\n|\r|\n)+", " | ")
                  .str.replace_all(r"[\-•\u2022\u25CF\u25E6\u2043\u2219\u00B7\u2013\u2014]+", "")
                  .str.replace_all(r"\s{2,}", " ")
                  .str.strip_chars()
            )
        }
    )
    .with_columns(
        pl.concat_str(
            [pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)],
            separator="_"
        ).alias("key_redirect")
    )
    .select(["key_redirect", "Re_Direct", "Re-Direct Text"])
    .unique(subset=["key_redirect"], maintain_order=True)
)

re_direct_final

key_redirect,Re_Direct,Re-Direct Text
str,i32,str
"""818815855_4c5b2004-ee05-4ec5-9…",1,"""Please note that the property …"
"""243121454_0c7d088c-48cd-4e00-8…",1,"""If you wish I can share our No…"
"""990726347_a7f305b0-bac8-4ea8-8…",1,"""So no need to worry about it, …"
"""981855762_ef9bf1c4-6337-452c-a…",1,"""however no worries i will prov…"
"""710538811_32b17761-6292-4fdd-a…",1,"""02 9148 3700 is their best num…"
…,…,…
"""937668880_63993d26-58db-4954-a…",1,"""sorry as If you are not able t…"
"""566504915_63993d26-58db-4954-a…",1,"""sorry as If you are not able t…"
"""437295389_63993d26-58db-4954-a…",1,"""sorry as If you are not able t…"


In [11]:
PERFORMANCE_INPUT = input_data(folder_paths["input_performance"], filename_keyword="aws_performance_retail_rawdata")


try:
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

print(PERFORMANCE_INPUT.columns)

# ── AWS schema adapter ──────────────────────────────────────────────────
# Maps the new AWS-export column names to the legacy canonical names the
# rest of the pipeline already expects, so everything downstream (HC join,
# AON, LC threshold, PST/VNT interval calc) stays untouched.
#
# ASSUMPTION — verify against the printed sample below before trusting the
# rest of the output:
#   • "Connected To Agent Date" = calendar date (e.g. 2026-06-01)
#   • "Connected To Agent Time" = either (a) time-of-day only (e.g.
#     08:23:45), combined with the Date column, or (b) already a full
#     "YYYY-MM-DD HH:MM:SS" datetime. The parser below auto-detects which
#     case applies, but it can only be confirmed against the real file.
#   • "Agent Vendor Location" is the successor of the old "Agent Business
#     Location" field (same role — feeds the final Ho Chi Minh site filter).

def _build_joined_time(time_col: str = "Connected To Agent Time") -> pl.Expr:
    raw = pl.col(time_col).cast(pl.Utf8).str.strip_chars()
    return (
        pl.coalesce([
            raw.str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),  # 2026-06-22 02:12:40
            raw.str.strptime(pl.Datetime, "%m/%d/%Y %H:%M",    strict=False),  # 6/22/2026 14:12 (máy cũ)
        ])
        .alias("Joined Time")
    )

def _duration_to_seconds(col: str) -> pl.Expr:
    """Cast a duration field to Float64 seconds. Handles either a plain
    numeric string (assumed already-in-seconds) or an 'HH:MM:SS' string."""
    raw       = pl.col(col).cast(pl.Utf8).str.strip_chars()
    as_number = raw.str.replace_all(",", "").cast(pl.Float64, strict=False)
    h = raw.str.extract(r"^(\d+):\d{2}:\d{2}$", 1).cast(pl.Float64, strict=False)
    m = raw.str.extract(r"^\d+:(\d{2}):\d{2}$", 1).cast(pl.Float64, strict=False)
    s = raw.str.extract(r"^\d+:\d{2}:(\d{2})$", 1).cast(pl.Float64, strict=False)
    from_hhmmss = h * 3600 + m * 60 + s
    return pl.when(as_number.is_not_null()).then(as_number).otherwise(from_hhmmss).alias(col)


PERFORMANCE_INPUT = (
    PERFORMANCE_INPUT
    .with_columns(_build_joined_time())
    .rename({
        "Handle Time":                    "Handle Time (Sum)",
        "Talk Time":                      "Talk Time (Sum)",
        "Acw Duration":                   "Wrap Up Time (Sum)",
        "Agent Vendor Location":          "Agent Business Location",
        "Outbound Initiated (Yes / No)":  "Initiated Outbound (Yes / No)",
        "Product":                        "Latest VA Product",
        "Intent":                         "Latest VA Intent",
        "Locale":                         "Language",
    })
    .with_columns([
        pl.col("Latest VA Product").fill_null("UNKNOWN").alias("Latest VA Product"),
        pl.col("Latest VA Intent").fill_null("UNKNOWN").alias("Latest VA Intent"),
    ])
)

# Verify the timestamp parsing before trusting anything downstream
print(
    PERFORMANCE_INPUT
    .select(["Connected To Agent Date", "Connected To Agent Time", "Joined Time"])
    .head(5)
)

existing_cols = set(PERFORMANCE_INPUT.columns)

# Only "Handle Time (Sum)" is actually used in a formula (_lc / Short Chat
# thresholds) further down; the other Float64 fields are cast too so the BI
# output keeps clean numeric types, matching original intent. "Handle (Count)"
# and "Hold Time (Sum)" exist on the raw file under their original names
# (confirmed) -- just need casting like before.
columns_to_cast = {
    "Handle Time (Sum)":  pl.Float64,
    "Talk Time (Sum)":    pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Hold Time (Sum)":    pl.Float64,
    "Handle (Count)":     pl.Int64,
}

casts = []
for _col, _dtype in columns_to_cast.items():
    if _col not in existing_cols:
        continue
    if _dtype == pl.Float64:
        casts.append(_duration_to_seconds(_col))
    else:
        casts.append(pl.col(_col).cast(_dtype, strict=False).alias(_col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)

LG_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Car_Activity",
    "Chat_AC_GLB_EN_Lodging_Nesting",
    "Chat_AC_GLB_EN_Lodging_Proficient",
]
NL_CHAT_PROFILES = [
    "Chat_AC_GLB_EN_Proficient",
    "Chat_AC_GLB_EN_NL_Nesting",
]

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Agent Routing Profile Name")
      .alias("Agent Queue Group Name"),
    pl.when(pl.col("Agent Routing Profile Name").is_in(LG_CHAT_PROFILES))
      .then(pl.lit("LG Chat"))
      .when(pl.col("Agent Routing Profile Name").is_in(NL_CHAT_PROFILES))
      .then(pl.lit("NL Chat"))
      .otherwise(pl.col("Agent Routing Profile Name"))
      .alias("LOB"),
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_NEXT_STEP.with_columns([
    (pl.col("Joined Time") + pl.duration(hours=14)).alias("Join Time (VNT)"),
    (pl.col("Joined Time") + pl.duration(hours=14)).dt.date().alias("Join Date (VNT)")
])

HC_MASTER_DATABASE = input_data(folder_paths["hc_extend_by_month"])
HC_MASTER_DATABASE = HC_MASTER_DATABASE.rename({'Date Start Week': 'Week_Monday'})

HC_MASTER_DATABASE = HC_MASTER_DATABASE.with_columns([
    pl.col('Date').str.strptime(pl.Date, "%Y-%m-%d", strict=False)
])

hc_master_selected = HC_MASTER_DATABASE.select([
    "Date","Email Id", "OracleID", "People ID", "IEX ID", "Employee Name", "Alias", "Designation", "Detail Status", "Active",
    "Supervisor Name", "Wave", "LOB", 'LG Tenure', 'NL Tenure', 'Mini TL - Email', 'Mini TL - Short Name', 'Mini TL Start Date', 'Site'
]).unique()

hc_master_selected = hc_master_selected.rename({'Mini TL - Short Name': 'Mini TL', 'LOB':'Group'})
performance_merged = PERFORMANCE_NEXT_STEP.join(
    hc_master_selected,
    left_on=["Joined Date","Agent Email ID"],
    right_on=["Date","Email Id"],
    how="left")
GLOBAL_HC = pl.read_parquet(folder_paths["global_hc"])
global_hc_clean = GLOBAL_HC.select(["SSO ID","Production Start date","Agent/Non Agent"]).unique(subset=["SSO ID"], keep="first")
merged_global_hc = performance_merged.join(global_hc_clean,left_on="Agent Email ID",right_on="SSO ID",how="left")
merged_iex = merged_global_hc.join(IEX[['Date','Email Id','First Shift','Datetime_First_Start_Shift','Night_Shift']], left_on=['Join Date (VNT)','Agent Email ID'], right_on=['Date','Email Id'], how='left')
mapping = pl.read_excel(folder_paths["mapping_file"])
performance_cleaned = merged_iex.join(mapping, on="Agent Queue Group Name",how='left')
lc_mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="kpi")

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Vendor Location', 'Outbound Initiated (Yes / No)', 'Business Segment Name', 'Partner Name', 'Locale', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time', 'Talk Time', 'Assigned Agent Time', 'Acw Duration', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Intent', 'Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time']
shape: (5, 3)
┌─────────────────────────┬─────────────────────────┬─────────────────────┐
│ Connected To Agent Date ┆ Connected To Agent Time ┆ Joined Time         │
│ ---                     ┆ ---                     ┆ ---        

In [12]:
performance_updated_ns = performance_cleaned.with_columns(
    (pl.col('Join Date (VNT)') - pl.duration(days=1)).alias('Previous Date')
).join(
    Night_Shift[['Date', 'Email Id', 'Night_Shift', 'Previous_Night_Shift']],
    left_on=['Join Date (VNT)', 'Agent Email ID'],
    right_on=['Date', 'Email Id'],
    how='left',
)


def update_night_shift(df: pl.DataFrame) -> pl.DataFrame:
    # Step 1: compute the correction flag AND update Night_Shift in one pass
    _ns_override = (
        (pl.col('Night_Shift') == 0) &
        (pl.col('Join Time (VNT)').dt.time() < pl.time(17, 0)) &
        (pl.col('Join Time (VNT)').dt.time() >= pl.time(0, 0)) &
        (pl.col('Previous_Night_Shift') == 1)
    )
    df = df.with_columns(
        # Night_Shift_2_Check: False means "this row spills from previous night shift"
        pl.when(_ns_override).then(False).otherwise(True).alias('Night_Shift_2_Check'),
        # Immediately update Night_Shift based on the same condition
        pl.when(_ns_override).then(1).otherwise(pl.col('Night_Shift')).alias('Night_Shift'),
    )

    # Step 2: compute _Date_Converted using the updated Night_Shift
    _hour = pl.col('Join Time (VNT)').dt.hour()
    df = df.with_columns(
        pl.when((_hour < 12) & (pl.col('Previous_Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        .when((_hour < 12) & (pl.col('Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)') - pl.duration(days=1))
        .when((_hour < 18) & (pl.col('Night_Shift') == 0))
          .then(pl.col('Join Date (VNT)'))
        .when((_hour >= 18) & (pl.col('Night_Shift') == 1))
          .then(pl.col('Join Date (VNT)'))
        .otherwise(pl.col('Join Date (VNT)'))
        .alias('_Date_Converted')
    )
    return df


performance_updated_ns = update_night_shift(performance_updated_ns)

In [13]:
# ── Group 1: flags + date/time dimensions + composite keys ──
# (Neutral/NPS score block and CCR72/_fup_72/_rr removed — no source columns)
_joined = pl.col("Joined Time")

performance_processed = (
    performance_updated_ns
    .with_columns(
        pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),
        _joined.dt.date().alias("_PST.Date"),
        _joined.dt.strftime("%y_%m").alias("_PST.Month"),
        _joined.dt.strftime("%G_%V").str.slice(2).alias("_PST.Week"),
        _joined.dt.year().alias("_PST.Year"),
        pl.concat_str([
            pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
            pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
            _joined.dt.strftime("%y%m%d%H%M%S"),
        ], separator="_").alias("_conver_unique"),
        pl.when(pl.col("Group").is_in(["Non_Lodging", "Lodging"])).then(pl.lit("agent")).otherwise(None).alias("Agent"),
        pl.col("_Date_Converted").alias("_Date"),
        pl.when(pl.col("Group") == "Lodging").then(pl.lit("LG Tenure"))
          .when(pl.col("Group") == "Non_Lodging").then(pl.lit("NL Tenure"))
          .otherwise(None).alias("Tenure"),
    )
)

# ── Group 2: AON_Days + AON Status + LC threshold + composite keys ──
# (_nps_type removed — depended on _promoter/_detractor/_neutral, which no longer exist)
performance_processed = (
    performance_processed
    .with_columns(
        (
            pl.col("_PST.Date").cast(pl.Date) -
            pl.col("Production Start date")
              .cast(pl.String)
              .str.to_date("%Y-%m-%d %H:%M:%S", strict=False)
              .cast(pl.Date)
        ).dt.total_days().cast(pl.Int32).alias("AON_Days"),
    )
    .with_columns(
        pl.when(
            pl.col("AON_Days").is_null() &
            pl.col("Agent/Non Agent").is_in(["Agent", "ID Deleted"])
        ).then(pl.lit("Nesting"))
          .when(pl.col("AON_Days") > 180).then(pl.lit("> 180 Days"))
          .when(pl.col("AON_Days") >= 91).then(pl.lit("91 - 180"))
          .when(pl.col("AON_Days") >= 61).then(pl.lit("61 - 90"))
          .when(pl.col("AON_Days") >= 31).then(pl.lit("31 - 60"))
          .when(pl.col("AON_Days") >= 0).then(pl.lit("00 - 30"))
          .otherwise(None).alias("AON Status"),
    )
    .join_asof(lc_mapping, left_on="_PST.Date", right_on="Effective Date", by="LOB", strategy="backward")
    .with_columns(
        (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc"),
        (pl.col("Handle Time (Sum)") < 240).cast(pl.Int8).alias("Short Chat"),
        pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY"),
        pl.concat_str([pl.col("Agent Email ID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("EmailID_ConversationID_KEY"),
        pl.concat_str([pl.col("OracleID").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("OracleID_ConversationID_KEY"),
        pl.concat_str([pl.col("Agent People Id").cast(pl.Utf8), pl.col("Conversation Id").cast(pl.Utf8)], separator="_").alias("PeopleID_ConversationID_KEY"),
    )
)

# ── Group 3: Time interval + period columns using native Polars ──
_pst_min  = pl.col("Joined Time").dt.minute()
_pst_hour = pl.col("Joined Time").dt.hour()
_vnt_min  = pl.col("Join Time (VNT)").dt.minute()
_vnt_hour = pl.col("Join Time (VNT)").dt.hour()
_shift_hour = pl.col("Datetime_First_Start_Shift").dt.hour()

def _interval_expr(hour_expr: pl.Expr, minute_expr: pl.Expr) -> pl.Expr:
    """Build HH:MM-HH:MM 30-minute bucket expression natively."""
    _start_min = pl.when(minute_expr < 30).then(pl.lit(0)).otherwise(pl.lit(30))
    _end_min   = pl.when(minute_expr < 30).then(pl.lit(29)).otherwise(pl.lit(59))
    return pl.concat_str([
        hour_expr.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        _start_min.cast(pl.Utf8).str.zfill(2), pl.lit("-"),
        hour_expr.cast(pl.Utf8).str.zfill(2), pl.lit(":"),
        _end_min.cast(pl.Utf8).str.zfill(2),
    ])

performance_processed = performance_processed.with_columns(
    _interval_expr(_pst_hour, _pst_min).alias("_PST.Interval"),
    _interval_expr(_vnt_hour, _vnt_min).alias("_VNT.Interval"),
    pl.when(_shift_hour >= 18).then(pl.lit("Night"))
      .when(_shift_hour >= 12).then(pl.lit("Mid"))
      .otherwise(pl.lit("Morning"))
      .alias("_VNT.Period"),
)

# ── Normalize Joined Time to string format for consistent joins downstream ──
def _normalize_jt(df: pl.DataFrame) -> pl.DataFrame:
    if "Joined Time" not in df.columns:
        return df
    return df.with_columns(
        pl.col("Joined Time")
          .cast(pl.Utf8)
          .str.replace("T", " ")
          .str.replace("Z", "")
          .str.slice(0, 19)
          .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
          .dt.strftime("%Y-%m-%d %H:%M:%S")
          .alias("Joined Time")
    )

# survey_ir / survey_ae / survey_duet / delayed_closure / verbatim / t3_final removed from this list
for _df_name in ["performance_processed", "t3_final", "RE_DIRECT_INPUT", "afcr_input"]:
    globals()[_df_name] = _normalize_jt(globals()[_df_name])

In [14]:
performance_combined = (
    performance_processed
    .join(t3_final,                                    left_on="EmailID_ConversationID_KEY", right_on="key_t3",               how="left")
    .join(delayed_closure,                             left_on="EmailID_ConversationID_KEY", right_on="key_delayed_closure",   how="left")
    .join(re_direct_final,                             left_on="PeopleID_ConversationID_KEY",right_on="key_redirect",          how="left")
    .join(survey_final.drop("Conversation Id"),        left_on="EmailID_ConversationID_KEY", right_on="key_survey",            how="left")
    .join(afcr_input,                                  on=["Agent Email ID", "Conversation Id"],                               how="left")
)

print(performance_combined.columns)
print(performance_combined.select(["_PST.Date", "Production Start date", "AON_Days"]).head())

selected_columns = [
    "Export Time", "File Name", "Agent People Id", "Business Segment Name", "Partner Name",
    "Response Count", "Response Time", "Latest VA Product", "Language", "Latest VA Intent", "Conversation Id",
    "Agent Queue Group Name", "Joined Time", "_PST.Interval", "Agent Email ID",
    "Handle (Count)", "Handle Time (Sum)", "Hold Time (Sum)", "Talk Time (Sum)",
    "Join Time (VNT)", "_VNT.Interval", "_VNT.Period", "Wrap Up Time (Sum)", "Agent Business Location",
    "_PST.Date", "_PST.Month", "_PST.Year", "_aob", "LOB", "_conver_unique",
    "Re_Direct", "Re-Direct Text", "T3",
    "Exceed Time", "Exceed Chat", "Exceed Bucket",
    "Agent Disconnect", "Ghost", "Requeued", "Traveler Unresponsive",
    "_nps_type", "_promoter", "_detractor", "_neutral", "_survey",
    "DUET", "_verbatim",
    "_PST.Week", "_lc", "AON Status", "Agent/Non Agent", "Tenure",
    "OracleID", "People ID", "IEX ID", "Employee Name", "Alias", "Designation",
    "Detail Status", "Active", "Supervisor Name", "Wave", "Group", "_Date",
    "Mini TL - Email", "Mini TL", "Mini TL Start Date", "Site",
    "Short Chat", "Passed Sessions", "Failed Sessions", "Total Sessions",
]

fcr_columns = [
    "LOB", "OracleID_ConversationID_KEY", "EmailID_ConversationID_KEY",
    "_PST.Week", "AON Status", "Agent/Non Agent", "Tenure", "Employee Name", "Alias",
    "Designation", "Detail Status", "Supervisor Name", "Wave", "Group", "_Date",
    "Mini TL - Email", "Mini TL", "Mini TL Start Date", "Site", "Agent Business Location",
]

missing_cols = [col for col in selected_columns if col not in performance_combined.columns]
print("Missing columns:", missing_cols)

performance_filtered = performance_combined.select(selected_columns)

performance_filtered = (
    performance_filtered
    .sort(by=["Conversation Id", "Agent Email ID", "Joined Time"], descending=[False, False, True])
    .with_columns(
        (pl.col("Joined Time").cum_count().over(["Conversation Id", "Agent Email ID"]) > 1)
          .cast(pl.Int8)
          .alias("Duplicate_Flag")
    )
    .unique()
)

['Connected To Agent Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Routing Profile Name', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Business Segment Name', 'Partner Name', 'Language', 'Connected To Agent Time', 'Agent Name', 'Agent Queue Group Name', 'Agent Manager Name', 'Handle Time (Sum)', 'Talk Time (Sum)', 'Assigned Agent Time', 'Wrap Up Time (Sum)', 'Offered', 'Requeued (Yes / No)', 'Response Time', 'Response Count', 'Actual Disconnect Reason', 'Requeue Time', 'Inbound Message Count', 'Outbound Message Count', 'Transfer Initiated (Yes / No)', 'Agent Tenure in Days', 'Initial Channel Type', 'Answer Time', 'Queue Time', 'Latest VA Intent', 'Latest VA Product', 'Contact Disconnect (Count)', 'Handle (Count)', 'Hold Time (Sum)', 'File Name', 'Export Time', 'Joined Time', 'LOB', 'Joined Date', 'Join Time (VNT)', 'Join Date (VNT)', 'OracleID', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Designation', 'Detail Status', 'Active', 'Supervisor

In [15]:
# Load the removal list from Excel
removed_id = pl.read_excel(folder_paths["mapping_file"], sheet_name="id_removed")

# Normalize Conversation Id: strip quotes + leading/trailing whitespace (native ops)
def clean_id(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.replace_all(r'[""\u201c\u201d]', "")  # remove curly + straight quotes
            .str.strip_chars()                           # native trim, faster than regex
    )

if "Conversation Id" not in removed_id.columns:
    raise ValueError("Sheet 'id_removed' must contain a 'Conversation Id' column.")

# Canonical removal set: normalized, non-null, non-empty, deduplicated
removed_id_norm = (
    removed_id
    .select(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
    .drop_nulls()
    .filter(pl.col("Conversation Id") != "")
    .unique()
)

# Flag matching rows instead of dropping them (preserves auditability)
performance_filtered = (
    performance_filtered
    .with_columns(clean_id(pl.col("Conversation Id")).alias("__conv_norm"))
    .join(
        removed_id_norm.select(pl.col("Conversation Id").alias("__conv_norm"))
                       .with_columns(pl.lit(1).alias("__rm")),
        on="__conv_norm",
        how="left",
    )
    .with_columns(pl.col("__rm").fill_null(0).alias("IDs Removed"))
    .drop(["__conv_norm", "__rm"])
)

# (Optional) quick summary stats
flagged = performance_filtered.select(pl.col("IDs Removed").sum().alias("flagged")).item()
total   = performance_filtered.height
print(f"Flagged (IDs Removed=1): {flagged} / {total} rows")

Flagged (IDs Removed=1): 0 / 549106 rows


In [16]:
performance_unique = performance_filtered.drop(["Export Time", "File Name"])
performance_unique = performance_unique.unique()

performance_hcm = performance_unique.filter(pl.col("Agent Business Location").str.contains("Ho Chi Minh", literal=True))
performance_all_site = performance_unique

In [17]:
# request_file_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/conversation_id.xlsx'
# df_request = pl.read_excel(request_file_path, engine='calamine')

# df_request = df_request.with_columns([
#     pl.col("Conversation Id").cast(pl.Utf8).str.strip_chars()
# ])

# perf_source = performance_all_site.with_columns([
#     ((pl.col("Handle Time (Sum)") + pl.col("Hold Time (Sum)") + pl.col("Wrap Up Time (Sum)")) /
#      pl.col("Handle (Count)")).alias("AHT_Calc")
# ])

# result_df = df_request.join(
#     perf_source,
#     on="Conversation Id",
#     how="left"
# )

# NOTE: this block still references removed Survey columns (_promoter,
# _detractor, _survey, DUET) — needs rework if ever reactivated. The
# AHT_Calc line above is fine as-is now that Hold Time (Sum) / Handle
# (Count) are back.
# final_output = result_df.select([
#     "Agent Email ID",
#     "Conversation Id",


#     pl.col("_promoter").alias("Promoter"),
#     pl.col("_detractor").alias("Detractor"),
#     pl.col("_survey").alias("Survey"),
#     "DUET",
#     pl.col("AHT_Calc").alias("AHT")
# ])

# print(f"Extracted {final_output.height} rows.")
# print(final_output.head())

# final_output.write_excel('extracted_metrics_report.xlsx')

In [18]:
# region EXPORT
output_dir = folder_paths["output_performance_combine"]

for (month_value,), group in performance_hcm.group_by(['_PST.Month'], maintain_order=True):
    base_name = str(month_value)
    group.write_csv(os.path.join(output_dir, f"{base_name}.csv"))
    group.write_parquet(os.path.join(output_dir, f"{base_name}_performance_en_vn.parquet"))

parquet_files = glob.glob(os.path.join(output_dir, "*_performance_en_vn.parquet"))
out_path_total = os.path.join(output_dir, "_performance_hcm.parquet")

lazy_frames = [pl.scan_parquet(f) for f in parquet_files]

(
    pl.concat(lazy_frames, how="diagonal_relaxed")
    .collect()
    .write_parquet(out_path_total)
)

#endregion

In [ ]:
print(performance_all_site['_PST.Month'].unique())

shape: (2,)
Series: '_PST.Month' [str]
[
	"26_06"
	"26_07"
]


In [20]:
print(performance_all_site.schema)
print(performance_all_site['_PST.Date'].unique())

Schema({'Agent People Id': String, 'Business Segment Name': String, 'Partner Name': String, 'Response Count': String, 'Response Time': String, 'Latest VA Product': String, 'Language': String, 'Latest VA Intent': String, 'Conversation Id': String, 'Agent Queue Group Name': String, 'Joined Time': String, '_PST.Interval': String, 'Agent Email ID': String, 'Handle (Count)': Int64, 'Handle Time (Sum)': Float64, 'Hold Time (Sum)': Float64, 'Talk Time (Sum)': Float64, 'Join Time (VNT)': Datetime(time_unit='us', time_zone=None), '_VNT.Interval': String, '_VNT.Period': String, 'Wrap Up Time (Sum)': Float64, 'Agent Business Location': String, '_PST.Date': Date, '_PST.Month': String, '_PST.Year': Int32, '_aob': Int32, 'LOB': String, '_conver_unique': String, 'Re_Direct': Int32, 'Re-Direct Text': String, 'T3': Int32, 'Exceed Time': Float64, 'Exceed Chat': Int64, 'Exceed Bucket': String, 'Agent Disconnect': Int64, 'Ghost': Int64, 'Requeued': Int64, 'Traveler Unresponsive': Int64, '_nps_type': Strin